# 확률분포 시 생성기 LoRA 파인튜닝 Colab 노트북

- base model: `Qwen/Qwen2.5-0.5B-Instruct`
- Colab T4 기준 QLoRA
- 입력: 사용자 경험 프롬프트
- 출력: 정확히 3줄 한국어 시
- 데이터셋: `data/train.jsonl`의 `messages` 형식
- 현재 기준 데이터셋: classic 200개 + modern 50개 = 250개
- adapter와 merged model을 모두 Hugging Face Hub에 업로드


## 1. 라이브러리 설치
최신 TRL의 `SFTConfig` 방식으로 학습한다.

In [1]:
!pip -q install -U "transformers>=4.56.0" "accelerate>=1.10.0" "peft>=0.17.0" "datasets>=4.0.0" "trl>=0.24.0" bitsandbytes huggingface_hub sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.3 MB/s eta 0:00:00


## 2. Hugging Face 로그인

In [2]:
from huggingface_hub import login
login()


## 3. 기본 설정

In [3]:
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO_ID = "z-unghyun/poem-generator-lora-adapter"
MERGED_REPO_ID = "z-unghyun/poem-generator-merged"
OUTPUT_DIR = "./outputs/poem-generator-lora"
MERGED_OUTPUT_DIR = "./outputs/poem-generator-merged"

# 기본값은 GitHub main의 data/train.jsonl을 직접 불러오는 방식이다.
# 로컬 파일을 직접 업로드하려면 DATA_MODE = "local_upload"로 바꾼다.
DATA_MODE = "github_raw"  # "github_raw", "local_upload", or "hub_dataset"
GITHUB_RAW_TRAIN_URL = "https://raw.githubusercontent.com/z-unghyun/poem-generator/main/data/train.jsonl"
LOCAL_TRAIN_PATH = "train.jsonl"
HF_DATASET_ID = "z-unghyun/poem-generator-dataset"
HF_DATA_FILE = "train.jsonl"
EXPECTED_ROWS = 250

LORA_TARGET_MODULES = ["q_proj", "v_proj"]
# 성능이 부족하면 아래처럼 확장해서 재학습한다.
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    BF16_AVAILABLE = torch.cuda.is_bf16_supported()
    print("bf16_available:", BF16_AVAILABLE)
else:
    raise RuntimeError("GPU가 잡히지 않았습니다. 런타임 유형을 T4 GPU로 바꿔주세요.")


cuda_available: True
gpu: Tesla T4
bf16_available: True


## 4. 데이터셋 불러오기

In [4]:
from datasets import load_dataset
from pathlib import Path

if DATA_MODE == "github_raw":
    dataset = load_dataset(
        "json",
        data_files=GITHUB_RAW_TRAIN_URL,
        split="train",
        download_mode="force_redownload",
    )
elif DATA_MODE == "local_upload":
    from google.colab import files
    if not Path(LOCAL_TRAIN_PATH).exists():
        uploaded = files.upload()
        if LOCAL_TRAIN_PATH not in uploaded:
            first_name = next(iter(uploaded.keys()))
            Path(LOCAL_TRAIN_PATH).write_bytes(uploaded[first_name])
            print(f"saved uploaded file as {LOCAL_TRAIN_PATH}")
    dataset = load_dataset("json", data_files=LOCAL_TRAIN_PATH, split="train")
elif DATA_MODE == "hub_dataset":
    dataset = load_dataset(HF_DATASET_ID, data_files=HF_DATA_FILE, split="train")
else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE}")

print(dataset)
print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 250
})
{'messages': [{'role': 'system', 'content': '너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다.'}, {'role': 'user', 'content': '경험: 강 근처에서 누군가를 기다리며 비탄을 느낌'}, {'role': 'assistant', 'content': '임이여, 물을 건너지 마오\n임은 끝내 물을 건너시네\n물에 빠져 죽으니 장차 어찌하리'}]}


## 5. 데이터셋 검수

In [5]:
def count_nonempty_lines(text: str) -> int:
    return len([line for line in str(text).splitlines() if line.strip()])

def validate_messages(example):
    messages = example.get("messages")
    if not isinstance(messages, list) or len(messages) != 3:
        return {"is_valid": False, "validation_reason": "messages_not_3_turns"}
    roles = [m.get("role") for m in messages]
    if roles != ["system", "user", "assistant"]:
        return {"is_valid": False, "validation_reason": f"invalid_roles:{roles}"}
    poem = messages[2].get("content", "")
    line_count = count_nonempty_lines(poem)
    if line_count != 3:
        return {"is_valid": False, "validation_reason": f"assistant_not_three_lines:{line_count}"}
    return {"is_valid": True, "validation_reason": "ok"}

checked = dataset.map(validate_messages)
bad = checked.filter(lambda x: not x["is_valid"])
print("total:", len(checked))
print("bad:", len(bad))

if len(checked) != EXPECTED_ROWS:
    raise ValueError(f"Expected exactly {EXPECTED_ROWS} rows, got {len(checked)}. data/train.jsonl이 최신 250개 버전인지 확인하세요.")
if len(bad) > 0:
    print(bad[:5])
    raise ValueError("Invalid training rows found. Fix train.jsonl before training.")

dataset = checked.remove_columns(["is_valid", "validation_reason"])


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Filter:   0%|          | 0/250 [00:00<?, ? examples/s]

total: 250
bad: 0


## 6. Tokenizer와 chat template 적용
`messages`를 Qwen chat template 문자열로 변환한 뒤, TRL에는 `text` 필드를 학습 대상으로 넘긴다.

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template)
print(dataset[0]["text"][:800])


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

<|im_start|>system
너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다.<|im_end|>
<|im_start|>user
경험: 강 근처에서 누군가를 기다리며 비탄을 느낌<|im_end|>
<|im_start|>assistant
임이여, 물을 건너지 마오
임은 끝내 물을 건너시네
물에 빠져 죽으니 장차 어찌하리<|im_end|>



## 7. QLoRA 모델 로드

In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if BF16_AVAILABLE else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 8. LoRA 설정 및 학습
최신 TRL 방식: `SFTConfig`에 `dataset_text_field`, `max_length`, `eos_token`을 넣고 `SFTTrainer`에는 `processing_class=tokenizer`를 넘긴다.

In [8]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=BF16_AVAILABLE,
    fp16=not BF16_AVAILABLE,
    optim="paged_adamw_8bit",
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_length=512,
    eos_token="<|im_end|>",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model = trainer.model
model.eval()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,4.741076
10,4.591854
15,4.168251
20,3.756741
25,3.292942
30,2.927226
35,2.711936
40,2.515399
45,2.523470
50,2.328631


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear4bi

## 9. Adapter 테스트

In [9]:
def make_prompt(experience: str):
    messages = [
        {"role": "system", "content": "너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다."},
        {"role": "user", "content": f"경험: {experience}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def clean_poem(text: str) -> str:
    text = text.strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines[:3])

def generate_poem(experience: str, max_new_tokens: int = 80):
    prompt = make_prompt(experience)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return clean_poem(generated)

test_inputs = [
    "비 오는 밤 버스 정류장에서 혼자 기다림",
    "새벽에 과제를 하다가 모니터 빛에 눈이 아픔",
    "보고 싶은 사람이 올 때를 기다리며 긴 밤을 견딤",
    "수영을 마치고 젖은 머리로 집에 돌아옴",
    "AI 프로젝트를 하다가 언어가 이상하게 무너지는 걸 봄",
]

for item in test_inputs:
    poem = generate_poem(item)
    print("경험:", item)
    print(poem)
    print("line_count:", count_nonempty_lines(poem))
    print("---")


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


경험: 비 오는 밤 버스 정류장에서 혼자 기다림
비 오는 데 있는 버스가 지나고
오늘은 휴식 시간이네요
버스는 지나가는길에 꿈을 가득한답니다
line_count: 3
---
경험: 새벽에 과제를 하다가 모니터 빛에 눈이 아픔
새벽에 과제가 한숨을 내다보니
모니터 빛이 젤런하고 있네
눈은 그 빛에 급진하네
line_count: 3
---
경험: 보고 싶은 사람이 올 때를 기다리며 긴 밤을 견딤
보고 싶은 사람이 밤에 고개를 높이 했으면
너는 길의 문이 밝아질 때마다
그 말들이 다시 곁에 서 있는지라
line_count: 3
---
경험: 수영을 마치고 젖은 머리로 집에 돌아옴
마른 빌드를 달고 간발을 탑입한
나무의 목소리는 낯겨 가득하고
이름당 길이 멀어 보인다
line_count: 3
---
경험: AI 프로젝트를 하다가 언어가 이상하게 무너지는 걸 봄
어민어 잠이 끝나 말은 늦은데
어민어 말이 안 되면 어민어 말이 오라
어민어 말이 없으면 말이 오라
line_count: 3
---


## 10. Adapter Hub 업로드

In [10]:
from huggingface_hub import create_repo

create_repo(ADAPTER_REPO_ID, exist_ok=True)
model.push_to_hub(ADAPTER_REPO_ID)
tokenizer.push_to_hub(ADAPTER_REPO_ID)


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  28%|##8       |  611kB / 2.18MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpicpkq6dd/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/z-unghyun/poem-generator-lora-adapter/commit/5e55099863dd592b5c8f7428ff723f8e32f02fe7', commit_message='Upload tokenizer', commit_description='', oid='5e55099863dd592b5c8f7428ff723f8e32f02fe7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/z-unghyun/poem-generator-lora-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='z-unghyun/poem-generator-lora-adapter'), pr_revision=None, pr_num=None)

## 11. Merged model 생성 및 업로드

In [13]:
# ## 11. Merged model 생성 및 업로드 - 최종 수정 버전

import sys
import subprocess
import importlib
import importlib.metadata as md
import gc
from pathlib import Path

# 1) torchao 버전 보정
def get_pkg_version(pkg_name: str):
    try:
        return md.version(pkg_name)
    except md.PackageNotFoundError:
        return None

torchao_version = get_pkg_version("torchao")
print("current torchao:", torchao_version)

if torchao_version is None or torchao_version < "0.16.0":
    print("Installing/upgrading torchao>=0.16.0 ...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "--no-cache-dir",
        "torchao>=0.16.0",
    ])

for name in list(sys.modules.keys()):
    if name == "torchao" or name.startswith("torchao."):
        del sys.modules[name]

importlib.invalidate_caches()
print("torchao after install:", get_pkg_version("torchao"))

# 2) imports
import torch
from huggingface_hub import create_repo, upload_folder
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 3) 기본값 복구
BASE_MODEL_ID = globals().get("BASE_MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MERGED_REPO_ID = globals().get("MERGED_REPO_ID", "z-unghyun/poem-generator-merged")
OUTPUT_DIR = globals().get("OUTPUT_DIR", "./outputs/poem-generator-lora")
MERGED_OUTPUT_DIR = globals().get("MERGED_OUTPUT_DIR", "./outputs/poem-generator-merged")

BF16_AVAILABLE = globals().get(
    "BF16_AVAILABLE",
    torch.cuda.is_available() and torch.cuda.is_bf16_supported()
)

if not Path(OUTPUT_DIR).exists():
    raise FileNotFoundError(
        f"{OUTPUT_DIR} 폴더가 없습니다. 8번 학습 셀 저장 결과가 없으니 8번 셀을 먼저 완료해야 합니다."
    )

# 4) 이미 merged 저장돼 있으면 merge 재생성 생략 가능
merged_dir = Path(MERGED_OUTPUT_DIR)
has_merged_files = (
    merged_dir.exists()
    and any(merged_dir.glob("*.safetensors"))
    and (merged_dir / "config.json").exists()
)

if has_merged_files:
    print("Existing merged model found. Skip merge and upload existing folder:", MERGED_OUTPUT_DIR)
else:
    # 메모리 정리
    try:
        del trainer
    except NameError:
        pass

    try:
        del model
    except NameError:
        pass

    gc.collect()
    torch.cuda.empty_cache()

    # tokenizer 준비
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # base model 로드
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.bfloat16 if BF16_AVAILABLE else torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    # LoRA adapter merge
    merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    merged_model = merged_model.merge_and_unload()

    # 로컬 저장
    merged_model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

    print("Merged model saved to:", MERGED_OUTPUT_DIR)

# 5) Hugging Face Hub 업로드
create_repo(MERGED_REPO_ID, exist_ok=True, repo_type="model")

upload_folder(
    repo_id=MERGED_REPO_ID,
    repo_type="model",
    folder_path=MERGED_OUTPUT_DIR,
    commit_message="Upload merged poem generator model",
)

print("Merged model uploaded to:", MERGED_REPO_ID)

current torchao: 0.17.0
torchao after install: 0.17.0
Existing merged model found. Skip merge and upload existing folder: ./outputs/poem-generator-merged
Merged model uploaded to: z-unghyun/poem-generator-merged


## 12. 다음 단계

학습이 끝나면 Hugging Face Space backend의 `MODEL_ID`를 `z-unghyun/poem-generator-merged`로 설정하고 `/generate` 응답을 확인한다.
